In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [12]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

# creates the client
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "arnaudon/netSALT",
    "PythonOT/POT",
    "blegat/SwitchOnSafety.jl",
    "connectomicslab/connectomemapper3",
    "BBN-Q/Auspex",
    "vsamy/Copra",
    "havakv/pycox",
    "seung-lab/igneous",
    "salan668/FAE",
    "matsengrp/transport",
]
list_df_commits = []
for prj in projects:
    # Convert GitHub repository names to WoC V2412 project names
    prj = prj.lower().replace("/", "_", 2)
    commits = woc.get_values("p2c", prj)
    df = pd.DataFrame(commits, columns=["sha1"])
    df["project"] = prj
    list_df_commits.append(df)
df = pd.concat(list_df_commits)
# perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv("df_commits.csv")
df.head(1)

,sha1,project
0,005227b76ce4b178c1e5cbf5d4ff2a92fed36456,arnaudon_netsalt


In [16]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 34%|███▍      | 579/1710 [10:17<20:27,  1.09s/it]

Got Errors {'4bb179ca211950c8b1499f8e78049b6e700f7ebf': 'Key 4bb179ca211950c8b1499f8e78049b6e700f7ebf not found in /da5_fast/All.sha1c/commit_75.tch'}


 35%|███▍      | 591/1710 [10:30<20:07,  1.08s/it]

Got Errors {'8738228d90a260a8b9b7ab99389bbb0551dfb2e2': 'Key 8738228d90a260a8b9b7ab99389bbb0551dfb2e2 not found in /da5_fast/All.sha1c/commit_7.tch'}


 69%|██████▉   | 1184/1710 [21:05<09:27,  1.08s/it]

...

Got Errors {'e5a29e8b97984a5464a62714a0fec119744df896': 'Key e5a29e8b97984a5464a62714a0fec119744df896 not found in /da5_fast/All.sha1c/commit_101.tch'}


 96%|█████████▋| 1646/1710 [29:27<01:09,  1.08s/it]

Got Errors {'580154d132f5d0f32cb763bcdcc37b8a9a3debcd': 'Key 580154d132f5d0f32cb763bcdcc37b8a9a3debcd not found in /da5_fast/All.sha1c/commit_88.tch'}


100%|██████████| 1710/1710 [30:36<00:00,  1.07s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,005227b76ce4b178c1e5cbf5d4ff2a92fed36456,276c366100a853ca08d7416179eeafaeb479666a,[048e05847a3c5ca2281cb7ba32bf1a8801d220bc],alexis arnaudon <alexis.arnaudon@epfl.ch>,1581848668,+0100,alexis arnaudon <alexis.arnaudon@epfl.ch>,1581848668,+0100,mode trajectories\n,005227b76ce4b178c1e5cbf5d4ff2a92fed36456,arnaudon_netsalt
1,0085f39eab2321e4b71da2cf97f286d79dcb4920,6797a539d79b970b158a2c32211976826bcc2579,[fcc8dfc53b8dc86a97457a247ab47861de52ff59],arnaudon <alexis.arnaudon@epfl.ch>,1646322150,+0100,arnaudon <alexis.arnaudon@epfl.ch>,1646322150,+0100,more\n,0085f39eab2321e4b71da2cf97f286d79dcb4920,arnaudon_netsalt


In [17]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '005227b76ce4b178c1e5cbf5d4ff2a92fed36456', 'tree': '276c366100a853ca08d7416179eeafaeb479666a', 'parent': ['048e05847a3c5ca2281cb7ba32bf1a8801d220bc'], 'author': 'alexis arnaudon <alexis.arnaudon@epfl.ch>', 'author_time': 1581848668, 'author_tz': '+0100', 'committer': 'alexis arnaudon <alexis.arnaudon@epfl.ch>', 'committer_time': 1581848668, 'committer_tz': '+0100', 'message': 'mode trajectories\n'}


In [18]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [19]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,matsengrp_transport,005227b76ce4b178c1e5cbf5d4ff2a92fed36456,alexis arnaudon <alexis.arnaudon@epfl.ch>,1581848668,mode trajectories\n


In [20]:
#mode a means append, so you have all your projects in the same file
yournetid='pshu'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github